# Домашнее задание 4. Линейный поиск, доверительная область, производные

**Выдано:** 21 октября 2026. **Срок сдачи:** 4 ноября 2026 (до начала занятия). **Максимум:** 10 баллов (+1 бонус).

**Главная идея.** Вдали от решения шаг Ньютона может увести куда угодно: квадратичная модель $m(p)=f(x)+g^\top p+\frac12p^\top Hp$ (здесь $g=\nabla f(x)$ — градиент, $H=\nabla^2f(x)$ — гессиан, $p$ — вектор шага) похожа на функцию только рядом с точкой $x$. Лекция 6 даёт два лекарства: проверить шаг самой функцией и укоротить его (линейный поиск, задача 1) или заранее ограничить шаг кругом, в котором модели можно верить (доверительная область, задача 2). Задача 3 — откуда берутся сами производные. Каждая задача заканчивается картинкой, и на вопросы вы отвечаете по ней.

**Как сдавать.** Работа сдаётся в ваш личный закрытый репозиторий `optimisation-misis-2026/numopt-<ваш логин>`; этот ноутбук уже лежит там как `hw4/homework.ipynb`. Код пишите в ячейках с `# TODO`, ответы — в markdown-ячейках вместо `_Ваш ответ:_ TODO`: 2–4 предложения с числами из ваших выводов. Выполните ноутбук целиком (`Kernel → Restart Kernel and Run All Cells`) и загрузите в папку `hw4/` под тем же именем — пошагово в [инструкции](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/CONTRIBUTING.md). Задачи можно обсуждать, но текст и код пишутся самостоятельно; своё решение никому не показывайте.

**Материал:** лекция 6 ([конспект](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/lectures/lecture06/lecture06.md), [демо](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/lectures/lecture06/demo06.ipynb), [семинар](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/lectures/lecture06/seminar06.ipynb)) — задачи 1–2 и бонус; лекция 7 (21 октября) — задача 3. Ничего сверх лекций не требуется. Ориентировочное время — 2 часа.

**Устройство задания.** Функции, их производные, подзадача доверительной области и рисовальщики даны готовыми в ячейке ниже. Вы пишете три короткие функции — `armijo`, `trust_region` и `fd_forward` с `fd_central`; под каждой стоит проверка `assert` на числах конспекта: прошла — можно идти дальше. Остальное — запуски, картинки и ответы.

Запуск: в Google Colab первая ячейка сама поставит CasADi; локально — `uv sync --group casadi` в корне клона репозитория курса, затем `uv run jupyter lab`.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("casadi") is None:          # Colab: поставить CasADi один раз
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "casadi"], check=True)

import casadi as ca
import numpy as np
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA, YELLOW, RED, VIOLET, GRAY, INK = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e34948", "#4a3aa7", "#8a8985", "#0b0b0b"
COLORS = [BLUE, ORANGE, AQUA, VIOLET]      # цвета путей по порядку: первый — синий, второй — оранжевый, ...
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.2, "legend.frameon": False})


# ---------------------------------------------------------------- задача 1: функция Диля, x = (u, v)
def f_diehl(x):
    """f(u, v) = 1/2 (u - 1)^2 + 1/2 v^2 + 5/2 (v - cos u)^2."""
    u, v = x
    return 0.5 * (u - 1) ** 2 + 0.5 * v ** 2 + 2.5 * (v - np.cos(u)) ** 2


def grad_diehl(x):
    u, v = x; r = v - np.cos(u)
    return np.array([(u - 1) + 5 * r * np.sin(u), v + 5 * r])


def hess_diehl(x):
    u, v = x; r = v - np.cos(u)
    return np.array([[1 + 5 * (np.sin(u) ** 2 + r * np.cos(u)), 5 * np.sin(u)],
                     [5 * np.sin(u), 6.0]])


X0_DIEHL = np.array([0.0, 10.0])            # старт
X_STAR_DIEHL = np.array([1.2496, 0.2631])   # минимум (для звёздочки на картинке)


# ---------------------------------------------------------------- Розенброк (как в лекциях)
def rosen(x):
    return (1 - x[0]) ** 2 + 100 * (x[1] - x[0] ** 2) ** 2


def rosen_grad(x):
    return np.array([-2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0] ** 2), 200 * (x[1] - x[0] ** 2)])


def rosen_hess(x):
    return np.array([[2 - 400 * x[1] + 1200 * x[0] ** 2, -400 * x[0]], [-400 * x[0], 200.0]])


X0_ROSEN = np.array([-1.2, 1.0])


# ---------------------------------------------------------------- два готовых метода Ньютона
def newton_pure(grad, hess, x0, tol=1e-6, maxit=100):
    """Чистый Ньютон (лекция 5): x <- x - H^{-1} g, шаг всегда полный. Возвращает (x, итераций, путь)."""
    x = np.asarray(x0, float).copy(); path = [x.copy()]
    for k in range(maxit):
        g = grad(x)
        if np.linalg.norm(g) <= tol:
            return x, k, np.array(path)
        x = x - np.linalg.solve(hess(x), g); path.append(x.copy())
    return x, maxit, np.array(path)


def newton_armijo(f, grad, hess, x0, tol=1e-6, maxit=100):
    """Ньютон, длину шага которого выбирает ВАШ armijo из задания 1. Возвращает (x, итераций, путь)."""
    x = np.asarray(x0, float).copy(); path = [x.copy()]
    for k in range(maxit):
        g = grad(x)
        if np.linalg.norm(g) <= tol:
            return x, k, np.array(path)
        p = -np.linalg.solve(hess(x), g)
        assert g @ p < 0, "направление Ньютона — не спуск (гессиан индефинитен): нужен сдвиг tau I, лекция 5"
        alpha, _ = armijo(f, x, g, p)
        x = x + alpha * p; path.append(x.copy())
    return x, maxit, np.array(path)


# ---------------------------------------------------------------- задача 2: подзадача доверительной области (конспект §3.3)
def cauchy_point(g, B, Delta):
    """Точка Коши: минимум модели g^T p + 1/2 p^T B p вдоль -g внутри круга радиуса Delta."""
    gBg = g @ B @ g; gn = np.linalg.norm(g)
    tau = 1.0 if gBg <= 0 else min(gn ** 3 / (Delta * gBg), 1.0)
    return -tau * Delta / gn * g


def tr_exact(g, B, Delta):
    """Точный шаг: минимум модели g^T p + 1/2 p^T B p в круге ||p|| <= Delta (Moré–Sorensen):
    p = -(B + lam I)^{-1} g с lam >= 0, B + lam I >= 0, lam (Delta - ||p||) = 0; lam — делением пополам."""
    lam, V = np.linalg.eigh(B); gt = V.T @ g

    def p_of(mu):
        return -V @ (gt / (lam + mu))
    if lam[0] > 0 and np.linalg.norm(p_of(0.0)) <= Delta:
        return p_of(0.0)                                         # точка Ньютона внутри круга
    lo = max(0.0, -lam[0]); hi = lo + np.linalg.norm(g) / Delta + 1.0
    if np.linalg.norm(p_of(lo + 1e-12 * max(1.0, lo))) < Delta:   # «трудный случай»
        p = p_of(lo + 1e-12 * max(1.0, lo))
        return p + np.sqrt(max(Delta ** 2 - p @ p, 0.0)) * V[:, 0]
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if np.linalg.norm(p_of(mid)) > Delta else (lo, mid)
    return p_of(hi)


# ---------------------------------------------------------------- задача 2(б): упругая цепь
def make_elastic_chain(N=40, L_total=5.0, D=70.0, g=9.81):
    """N грузиков массой 4/N между концами (-2, 1) и (2, 1); N + 1 пружин жёсткости D с длиной покоя L_total / (N + 1).
    v = (y_1..y_N, z_1..z_N): сначала горизонтальные координаты, потом вертикальные.
    E(v) = D/2 sum (d_j - L)^2 + m g sum z_i, d_j — длины пружин. Возвращает (E, grad E, гессиан E, старт — прямая)."""
    m = 4.0 / N; L = L_total / (N + 1)

    def energy(v):
        y, z = chain_points(v); d = np.hypot(np.diff(y), np.diff(z))
        return 0.5 * D * np.sum((d - L) ** 2) + m * g * np.sum(z[1:-1])

    def grad(v):
        y, z = chain_points(v); dy, dz = np.diff(y), np.diff(z); c = D * (1 - L / np.hypot(dy, dz))
        return np.r_[c[:-1] * dy[:-1] - c[1:] * dy[1:], c[:-1] * dz[:-1] - c[1:] * dz[1:] + m * g]

    def hess(v):
        y, z = chain_points(v); dy, dz = np.diff(y), np.diff(z); d = np.hypot(dy, dz)
        a = D * (1 - L / d); w = D * L / d ** 3
        Kyy, Kzz, Kyz = a + w * dy * dy, a + w * dz * dz, w * dy * dz        # блок 2x2 каждой пружины
        H = np.zeros((2 * N, 2 * N)); idx = np.arange(N)
        for (i0, j0), K in (((0, 0), Kyy), ((N, N), Kzz), ((0, N), Kyz), ((N, 0), Kyz)):
            H[i0 + idx, j0 + idx] += K[:-1] + K[1:]
            H[i0 + idx[:-1], j0 + idx[1:]] -= K[1:-1]
            H[i0 + idx[1:], j0 + idx[:-1]] -= K[1:-1]
        return H

    return energy, grad, hess, np.r_[np.linspace(-2, 2, N + 2)[1:-1], np.ones(N)]


def chain_points(v):
    """Координаты всех точек цепи вместе с концами: (y, z)."""
    N = len(v) // 2
    return np.r_[-2.0, v[:N], 2.0], np.r_[1.0, v[N:], 1.0]


chain_E, chain_grad, chain_hess, V0 = make_elastic_chain()


# ---------------------------------------------------------------- рисовальщики
def _fraction(t):
    return "1" if t == 1 else r"$\frac{1}{%d}$" % round(1 / t)


def plot_phi(f, x, g, p, tried, H=None, c1=1e-4, ax=None):
    """Картинка 1. Разрез phi(alpha) = f(x + alpha p) на [0, 1], прямая Армихо, модель Ньютона вдоль луча
    (если дан гессиан H) и испытанные шаги tried: отвергнутые — красные крестики, принятый (последний) — зелёный круг."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7.5, 4.4))
    a = np.linspace(0, 1.05 * max(tried), 400); f0, slope = f(x), g @ p
    ax.plot(a, [f(x + t * p) for t in a], color=BLUE, lw=2, label=r"функция вдоль шага $\varphi(\alpha)$")
    ax.plot(a, f0 + c1 * a * slope, "--", color=GRAY, lw=1.6, label=r"прямая Армихо $f(x)+c_1\alpha\, g^\top p$")
    if H is not None:
        ax.plot(a, f0 + a * slope + 0.5 * a ** 2 * (p @ H @ p), ":", color=ORANGE, lw=2.2, label=r"модель Ньютона $m(\alpha p)$")
    for t in tried:
        ok = t == tried[-1]; ft = f(x + t * p)
        ax.plot(t, ft, "o" if ok else "x", color=AQUA if ok else RED, ms=9 if ok else 10, mew=2.2, mec="white" if ok else RED, zorder=6)
        ax.annotate(r"$\alpha=$" + _fraction(t), (t, ft), textcoords="offset points", xytext=(-46, 6) if t > 0.8 else (6, 8), fontsize=9, color=INK)
    ax.plot([], [], "x", color=RED, ms=8, mew=2, label="отвергнутые шаги"); ax.plot([], [], "o", color=AQUA, ms=8, label="принятый шаг")
    ax.axhline(0, color=INK, lw=0.6)
    ax.set(xlabel=r"$\alpha$", ylabel="значение", title="Функция и модель вдоль шага")
    ax.legend(fontsize=8, loc="upper left")
    return ax


def plot_paths_diehl(paths, labels, zoom=((-5.0, 6.5), (-3.5, 10.8))):
    """Картинка 2, слева и в центре. Пути методов на линиях уровня функции Диля: весь путь и окрестность минимума."""
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
    pts = np.vstack(paths); lo, hi = pts.min(0), pts.max(0); pad = 0.06 * (hi - lo) + 0.5
    windows = (((lo[0] - pad[0], hi[0] + pad[0]), (lo[1] - pad[1], hi[1] + pad[1])), zoom)
    for ax, (xl, yl), title in zip(axes, windows, ("Весь путь", "Окрестность минимума")):
        U, V = np.meshgrid(np.linspace(*xl, 400), np.linspace(*yl, 400))
        ax.contour(U, V, f_diehl(np.array([U, V])), levels=np.geomspace(0.1, 4000, 16), colors=GRAY, linewidths=0.6, alpha=0.7)
        for i, (P, lab) in enumerate(zip(paths, labels)):
            ax.plot(P[:, 0], P[:, 1], "-o", color=COLORS[i], ms=4, lw=1.4, label=f"{lab}, шагов: {len(P) - 1}")
        ax.plot(*paths[0][0], "s", color=INK, ms=7, zorder=7)
        ax.plot(*X_STAR_DIEHL, "*", color=RED, ms=15, mec="white", zorder=8)
        ax.set(xlim=xl, ylim=yl, xlabel="$u$", ylabel="$v$", title=title); ax.grid(False)
    axes[0].legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    return axes


def plot_values(values, labels, ax=None):
    """Картинка 2, справа. Значения f(x_k) по итерациям для нескольких методов (лог. шкала)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4))
    for i, (vals, lab) in enumerate(zip(values, labels)):
        ax.semilogy(np.arange(len(vals)), vals, "-o", color=COLORS[i], ms=4, lw=1.5, label=lab)
    ax.set(xlabel="итерация $k$", ylabel="$f(x_k)$", title="Значение функции по итерациям")
    ax.legend(fontsize=8)
    return ax


def plot_tr(hist, path, xlim=(-2.0, 2.0), ylim=(-1.0, 3.0)):
    """Картинка 3. Слева — путь доверительной области на линиях уровня Розенброка: круги радиуса Delta_k вокруг x_k,
    отвергнутые пробные точки x_k + p_k — красные крестики. Справа — Delta_k и rho_k по итерациям
    (rho_k вне [-1, 1.6] прижаты к краю). hist — история из вашей trust_region, path — её путь."""
    fig = plt.figure(figsize=(12.5, 5.2))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.1, 1], hspace=0.4, wspace=0.25)
    ax = fig.add_subplot(gs[:, 0]); a1 = fig.add_subplot(gs[0, 1]); a2 = fig.add_subplot(gs[1, 1], sharex=a1)
    U, V = np.meshgrid(np.linspace(*xlim, 400), np.linspace(*ylim, 400))
    ax.contour(U, V, rosen(np.array([U, V])), levels=np.geomspace(0.3, 2000, 14), colors=GRAY, linewidths=0.6, alpha=0.7)
    for xk, pk, Dk, rk, ok in hist:
        ax.add_patch(plt.Circle(xk, Dk, fill=False, color=BLUE, lw=0.8, alpha=0.3))
        if not ok:
            ax.plot(*(xk + pk), "x", color=RED, ms=10, mew=2.2, zorder=7)
    ax.plot(path[:, 0], path[:, 1], "-o", color=BLUE, ms=3.5, lw=1.4, zorder=6, label=f"принятые точки, шагов: {len(path) - 1}")
    ax.plot([], [], "x", color=RED, ms=8, mew=2, label="отвергнутые пробные точки")
    ax.plot(1, 1, "*", color=RED, ms=15, mec="white", zorder=8)
    ax.set(xlim=xlim, ylim=ylim, aspect="equal", xlabel="$x_1$", ylabel="$x_2$", title="Путь и круги доверия"); ax.grid(False)
    ax.legend(fontsize=8, loc="upper right")
    k = np.arange(len(hist)); D = np.array([h[2] for h in hist]); rho = np.array([h[3] for h in hist]); ok = np.array([h[4] for h in hist])
    a1.step(k, D, where="mid", color=BLUE, lw=2); a1.set_yscale("log", base=2)
    a1.plot(k[~ok], D[~ok], "x", color=RED, ms=9, mew=2.2)
    a1.set(ylabel=r"радиус $\Delta_k$", title="Радиус и отношение по итерациям")
    for level, txt, va in ((0.75, r"$\frac{3}{4}$", "center"), (0.25, r"$\frac{1}{4}$", "bottom"), (0.1, r"$\eta=0.1$", "top")):
        a2.axhline(level, color=GRAY, ls=":", lw=1); a2.text(k[-1] + 0.8, level, txt, va=va, fontsize=10, color=GRAY)
    rc = np.clip(rho, -1, 1.6)
    a2.plot(k[ok], rc[ok], "o", color=BLUE, ms=5, label="шаг принят")
    a2.plot(k[~ok], rc[~ok], "x", color=RED, ms=9, mew=2.2, label="шаг отвергнут")
    a2.set(xlabel="итерация $k$", ylabel=r"отношение $\rho_k$", ylim=(-1.1, 1.7), xlim=(-1, k[-1] + 4.5))
    a2.legend(fontsize=8, loc="lower right")
    return ax, a1, a2


def plot_chains(chains, labels, start=None):
    """Картинка 4. Слева — цепи целиком, справа — низ цепи крупно. chains — список векторов v, labels — подписи."""
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), gridspec_kw={"width_ratios": [1.35, 1]})
    bottoms = []
    for ax in axes:
        if start is not None:
            y, z = chain_points(start); ax.plot(y, z, "--", color=GRAY, lw=1.2, label="старт: прямая")
        for i, (v, lab) in enumerate(zip(chains, labels)):
            y, z = chain_points(v); ax.plot(y, z, "-o", color=COLORS[i], ms=3.5, lw=1.4, label=lab)
            if ax is axes[0]:
                bottoms.append((y[np.argmin(z)], z.min()))
        ax.plot([-2, 2], [1, 1], "s", color=INK, ms=7, zorder=7)
        ax.set(aspect="equal", xlabel="$y$", ylabel="$z$")
    yc, zc = np.mean(bottoms, axis=0)
    axes[0].set_title("Форма цепи"); axes[0].legend(fontsize=8, loc="center")
    axes[1].set(xlim=(yc - 0.35, yc + 0.35), ylim=(zc - 0.12, zc + 0.3), title="Низ цепи крупно")
    plt.tight_layout()
    return axes


def plot_ucurve(hs, errors, labels, ad_error=None, ax=None):
    """Картинка 5. Ошибка разностной производной по шагу h (log-log); пунктир — sqrt(eps) и eps^(1/3);
    ad_error — ошибка CasADi (горизонталь; ноль рисуется на уровне 1e-16)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7.5, 4.6))
    eps = np.finfo(float).eps
    for i, (e, lab) in enumerate(zip(errors, labels)):
        ax.loglog(hs, np.maximum(np.asarray(e), 1e-17), color=COLORS[i], lw=1.8, label=lab)
    for h0, txt in ((np.sqrt(eps), r"$\sqrt{\varepsilon}$"), (eps ** (1 / 3), r"$\varepsilon^{1/3}$")):
        ax.axvline(h0, color=GRAY, ls=":", lw=1.2); ax.text(h0 * 1.3, 3e-16, txt, fontsize=10, color=GRAY)
    if ad_error is not None:
        ax.axhline(max(ad_error, 1e-16), color=AQUA, lw=2.2, label="CasADi: ошибка " + ("0" if ad_error == 0 else f"{ad_error:.0e}"))
    ax.set(xlabel="шаг $h$", ylabel="ошибка производной", ylim=(1e-17, 10), title="Ошибка разностной производной")
    ax.legend(fontsize=8, loc="upper center")
    return ax


def plot_subproblem(g, B, Delta, steps, ax=None):
    """Бонус. Линии уровня модели g^T p + 1/2 p^T B p, круг радиуса Delta, ломаная dogleg 0 -> p_U -> p_B
    и шаги steps = {подпись: p}."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6.2, 5))
    pB = -np.linalg.solve(B, g); pU = -(g @ g) / (g @ B @ g) * g
    r = 1.25 * max(np.linalg.norm(pB), Delta)
    U, V = np.meshgrid(np.linspace(-r, r, 300), np.linspace(-r, r, 300))
    M = g[0] * U + g[1] * V + 0.5 * (B[0, 0] * U ** 2 + 2 * B[0, 1] * U * V + B[1, 1] * V ** 2)
    ax.contour(U, V, M, levels=12, colors=GRAY, linewidths=0.6, alpha=0.7)
    ax.add_patch(plt.Circle((0, 0), Delta, fill=False, color=INK, lw=1.4))
    ax.plot([0, pU[0], pB[0]], [0, pU[1], pB[1]], "--", color=GRAY, lw=1.5, label="ломаная dogleg")
    ax.plot(*pB, "P", color=INK, ms=9, label="точка Ньютона $p_B$")
    m = lambda p: g @ p + 0.5 * p @ B @ p
    for i, (lab, p) in enumerate(steps.items()):
        ax.plot(*p, "o", color=COLORS[i], ms=9, mec="white", mew=1.5, zorder=7, label=f"{lab}: m = {m(p):.2f}")
    ax.plot(0, 0, "s", color=INK, ms=6)
    ax.set(aspect="equal", xlabel="$p_1$", ylabel="$p_2$", title=f"Подзадача при Δ = {Delta}"); ax.grid(False)
    ax.legend(fontsize=8, loc="upper right")
    return ax


print(f"функция Диля в старте: f = {f_diehl(X0_DIEHL):.1f};  цепь в старте: E = {chain_E(V0):.2f}, "
      f"lambda_min = {np.linalg.eigvalsh(chain_hess(V0))[0]:.1f}")

## Задача 1. Линейный поиск: Ньютон с правилом Армихо (3.5 балла)

**Главная идея.** Правило Армихо проверяет шаг самой функцией: шаг $\alpha p$ принимается, только если $f$ уменьшилась хотя бы на малую долю того, что обещает наклон $g^\top p$; иначе $\alpha$ делится пополам. Так длинный шаг Ньютона, которому модель «поверила» напрасно, укорачивается до разумного.

Функция — по мотивам упражнений курса NUMOPT М. Диля (Фрайбург):

$$
f(u,v)=\tfrac12(u-1)^2+\tfrac12v^2+\tfrac52\,(v-\cos u)^2 ,
$$

сумма квадратов трёх невязок; третья штрафует отход от кривой $v=\cos u$. Точка — $x=(u,v)$, старт $x_0=(0,\,10)$, минимум $x^\ast\approx(1.250,\,0.263)$. Функция, градиент и гессиан даны: `f_diehl`, `grad_diehl`, `hess_diehl`. Готовы и оба метода: `newton_pure` — чистый Ньютон, шаг всегда полный; `newton_armijo` — Ньютон, длину шага которого выбирает ваш `armijo`. Оба останавливаются при $\lVert\nabla f\rVert\le10^{-6}$.

**Задание 1 (1 балл): `armijo(f, x, g, p, c1=1e-4)`** — backtracking по Армихо (конспект §2.1). Начните с $\alpha=1$ и делите его пополам, пока не выполнено

$$
f(x+\alpha p)\le f(x)+c_1\,\alpha\,g^\top p ,\qquad c_1=10^{-4}.
$$

Верните принятый шаг $\alpha$ и список **всех** испытанных $\alpha$ по порядку (принятый — последний): список нужен для картинки.

In [ ]:
def armijo(f, x, g, p, c1=1e-4):
    """Backtracking по Армихо от alpha = 1 делением пополам. Возвращает (alpha, список испытанных alpha)."""
    # TODO: ваш код
    raise NotImplementedError

In [ ]:
# проверка задания 1: Розенброк в точке после первого шага Ньютона из (-1.2, 1) — конспект §2.1
xr = X0_ROSEN - np.linalg.solve(rosen_hess(X0_ROSEN), rosen_grad(X0_ROSEN))
gr = rosen_grad(xr); pr = -np.linalg.solve(rosen_hess(xr), gr)
alpha, tried = armijo(rosen, xr, gr, pr)
assert list(tried) == [1.0, 0.5, 0.25, 0.125] and alpha == 0.125, \
    f"в конспекте испытаны 1, 1/2, 1/4, 1/8 и принят 1/8, а у вас {tried}: начните с 1, делите пополам и сравнивайте с f(x) + c1 alpha g^T p"
print("задание 1: ok;  точка", xr.round(3), " испытаны", tried)

**(а) Картинка 1 (1 балл): почему полный шаг плох.** Сделайте один шаг Ньютона из $x_0$ — получите точку $x_1$. В ней вычислите градиент $g_1$, гессиан $H_1$ и шаг Ньютона $p_1=-H_1^{-1}g_1$; вызовите ваш `armijo`. Нарисуйте `plot_phi(f_diehl, x1, g1, p1, tried1, H=H1)`: функция вдоль шага $\varphi(\alpha)=f(x_1+\alpha p_1)$, прямая Армихо, модель Ньютона вдоль того же луча и испытанные шаги.

Ответьте: какие $\alpha$ отвергнуты и какой принят? Во сколько раз полный шаг увеличивает $f$ и что обещала модель? Найдите собственные числа $H_1$ (`np.linalg.eigh`) и длину $\lVert p_1\rVert$: почему шаг Ньютона здесь такой длинный?

*Полный балл:* картинка; испытанные шаги и значения $f$; собственные числа $H_1$ и $\lVert p_1\rVert$; объяснение длины шага через собственные числа. *Половина:* картинка и числа без объяснения.

In [ ]:
# TODO: ваш код
# x1 — точка после одного шага Ньютона из X0_DIEHL; g1, H1 — градиент и гессиан в x1; p1 = -H1^{-1} g1
# alpha1, tried1 = armijo(f_diehl, x1, g1, p1); напечатайте f(x1), f(x1 + p1), собственные числа H1 и |p1|
# картинка: plot_phi(f_diehl, x1, g1, p1, tried1, H=H1); plt.show()


_Ваш ответ:_ TODO

**(б) Картинка 2 (1.5 балла): чистый Ньютон против Ньютона с Армихо.** Запустите `newton_pure` и `newton_armijo` из $x_0$. Посчитайте для каждого число итераций, сколько раз $f$ выросла за шаг и максимум $f$ на пути. Нарисуйте пути — `plot_paths_diehl([path_pure, path_armijo], labels)`, слева весь путь, справа окрестность минимума, — и значения $f(x_k)$ по итерациям: `plot_values(...)`. Посчитайте $\lambda_{\min}$ (наименьшее собственное число гессиана) в точках обоих путей.

Ответьте: куда уходит чистый Ньютон и почему $f$ у него растёт? Что означает $\lambda_{\min}<0$ для модели и куда тогда ведёт шаг Ньютона? Почему у Ньютона с Армихо $f$ убывает на каждом шаге?

*Полный балл:* обе картинки; числа итераций, подъёмов и максимума; объяснение подъёмов через $\lambda_{\min}$ на пути и монотонности — через условие Армихо. *Половина:* запуски и картинки без объяснения.

In [ ]:
# TODO: ваш код
# x_pure, k_pure, path_pure = newton_pure(grad_diehl, hess_diehl, X0_DIEHL) и то же для newton_armijo
# посчитайте подъёмы f, максимум f и lambda_min вдоль путей (np.linalg.eigvalsh)
# картинки: plot_paths_diehl([...], [...]); plt.show()  и  plot_values([...], [...]); plt.show()


_Ваш ответ:_ TODO

## Задача 2. Доверительная область (4 балла)

**Главная идея.** Доверительная область делает наоборот: сначала решает, в каком радиусе $\Delta$ модели можно верить, и берёт лучшую точку модели в круге $\lVert p\rVert\le\Delta$. Радиус настраивается сам — по тому, насколько модель угадала:

$$
\rho_k=\frac{f(x_k)-f(x_k+p_k)}{m_k(0)-m_k(p_k)}=\frac{\text{фактическое уменьшение}}{\text{обещанное моделью}},\qquad m_k(0)-m_k(p_k)=-\Big(g_k^\top p_k+\tfrac12\,p_k^\top B_kp_k\Big).
$$

Здесь $B_k=\nabla^2f(x_k)$ — гессиан в текущей точке. Как найти лучшую точку модели в круге (*подзадача*), решено за вас: `tr_exact(g, B, Delta)` — точный шаг (конспект §3.3), `cauchy_point(g, B, Delta)` — точка Коши.

**Задание 2 (1.5 балла): `trust_region(f, grad, hess, x0, solve_sub)`** — цикл по алгоритму конспекта §3.2. Для $k=0,1,2,\dots$:

1. $g_k=\nabla f(x_k)$, $B_k=\nabla^2f(x_k)$; если $\lVert g_k\rVert\le10^{-6}$ — стоп;
2. $p_k$ = `solve_sub(g_k, B_k, Delta_k)`;
3. $\rho_k$ — по формуле выше (одно новое значение $f$);
4. радиус: при $\rho_k<\tfrac14$ — $\Delta_{k+1}=\Delta_k/4$; при $\rho_k>\tfrac34$ и шаге на границе ($\lVert p_k\rVert\ge0.99\,\Delta_k$) — $\Delta_{k+1}=\min(2\Delta_k,\ \Delta_{\max})$; иначе $\Delta_{k+1}=\Delta_k$;
5. шаг: при $\rho_k>\eta$ — $x_{k+1}=x_k+p_k$ (принят), иначе $x_{k+1}=x_k$ (отвергнут).

Параметры: $\Delta_0=1$, $\Delta_{\max}=10$, $\eta=0.1$. Итерация — один проход шагов 1–5, отвергнутые тоже считаются. Верните `(x, число итераций, путь, история)`: путь — массив принятых точек, начиная с $x_0$; история — список `(x_k, p_k, Delta_k, rho_k, принят)` по всем итерациям, его читает рисовальщик.

In [ ]:
def trust_region(f, grad, hess, x0, solve_sub, Delta=1.0, Delta_max=10.0, eta=0.1, tol=1e-6, maxit=1000):
    """Доверительная область (конспект §3.2). Возвращает (x, итераций, путь, история [(x_k, p_k, Delta_k, rho_k, принят)])."""
    # TODO: ваш код
    raise NotImplementedError

In [ ]:
# проверка задания 2: Розенброк из (-1.2, 1) с точным шагом — 25 итераций, 3 отвергнуты (конспект §3.4)
x_tr, k_tr, path_tr, hist_tr = trust_region(rosen, rosen_grad, rosen_hess, X0_ROSEN, tr_exact)
n_rej = sum(not h[4] for h in hist_tr)
assert np.allclose(x_tr, [1, 1], atol=1e-5), f"метод должен прийти в минимум (1, 1), а пришёл в {x_tr}"
assert (k_tr, n_rej) == (25, 3), \
    f"в конспекте 25 итераций и 3 отвергнутых шага, у вас {k_tr} и {n_rej}: сверьте правила радиуса, порог «на границе» и счёт отвергнутых"
print(f"задание 2: ok;  итераций {k_tr}, отвергнуто {n_rej}")

**(а) Картинка 3 (1 балл): как дышит радиус.** Нарисуйте `plot_tr(hist_tr, path_tr)` для запуска из проверки: слева путь на линиях уровня Розенброка и круги $\Delta_k$ вокруг $x_k$, отвергнутые пробные точки $x_k+p_k$ — красные крестики; справа $\Delta_k$ и $\rho_k$ по итерациям. Сравните с [картинкой конспекта](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/lectures/lecture06/img/06_tr_rosenbrock.png).

Ответьте: на каких итерациях шаги отвергнуты и каким было $\rho_k$? Что после отказа делает радиус и почему следующая попытка из той же точки удаётся? Когда радиус растёт и почему у решения он больше не мешает?

*Полный балл:* картинка; номера отвергнутых итераций и их $\rho_k$; что делает радиус после отказа и после удачного шага; почему у решения шаги ньютоновские. *Половина:* картинка без разбора.

In [ ]:
# TODO: ваш код
# номера отвергнутых итераций и их rho_k — из hist_tr; картинка: plot_tr(hist_tr, path_tr); plt.show()


_Ваш ответ:_ TODO

**(б) Картинка 4 (1.5 балла): равновесие или минимум?** Упругая цепь: $N=40$ грузиков между закреплёнными концами $(-2,\,1)$ и $(2,\,1)$, соседи соединены пружинами; энергия $E$ — упругая плюс потенциальная. Длина пружин в покое в сумме $5$, а расстояние между концами $4$, поэтому в стартовой прямой пружины сжаты, и она неустойчива: у гессиана энергии в ней 40 отрицательных собственных чисел. Даны `chain_E`, `chain_grad`, `chain_hess` и старт `V0` — вектор координат грузиков: сначала 40 горизонтальных, потом 40 вертикальных.

Запустите из `V0` чистый Ньютон `newton_pure` и вашу `trust_region` с `tr_exact`. Для обоих ответов напечатайте число итераций, $E$, $\lVert\nabla E\rVert$ и $\lambda_{\min}$ гессиана; нарисуйте обе цепи — `plot_chains([v_newton, v_tr], labels, start=V0)`, справа низ цепи крупно.

Ответьте: оба ли ответа — равновесия ($\nabla E=0$)? Какое из них устойчиво и как это видно по $\lambda_{\min}$ и по правой панели? Почему чистый Ньютон не отличает минимум от седла, а доверительная область отличает?

*Полный балл:* таблица из двух строк и картинка; какое равновесие устойчиво — по $\lambda_{\min}$ и по картинке; объяснение разницы между методами. *Половина:* числа и картинка без объяснения.

In [ ]:
# TODO: ваш код
# v_newton, k_newton, _ = newton_pure(chain_grad, chain_hess, V0);  v_tr, k, _, hist = trust_region(..., tr_exact)
# для обоих: итерации, chain_E(v), норма chain_grad(v), np.linalg.eigvalsh(chain_hess(v))[0]
# картинка: plot_chains([v_newton, v_tr], ["чистый Ньютон", "доверительная область"], start=V0); plt.show()


_Ваш ответ:_ TODO

## Задача 3. Производные: конечные разности и CasADi (2.5 балла)

**Главная идея.** Производную можно приблизить разностью значений: $f'(u)\approx\dfrac{f(u+h)-f(u)}{h}$. У такой формулы две ошибки. **Усечения** — формула неточна, ошибка растёт с $h$. **Округления** — при малом $h$ вычитаются почти равные числа, и относительная точность чисел $\varepsilon\approx2.2\cdot10^{-16}$ (машинная точность) превращается в ошибку порядка $\varepsilon\lvert f\rvert/h$. Сумма двух ошибок даёт U-образную кривую с лучшим $h$ посередине. Алгоритмическое дифференцирование (CasADi) применяет правила дифференцирования к каждой операции программы — без $h$ и без U.

Берём $\partial f/\partial u$ функции из задачи 1 в точке $x=(1,\,1)$; точное значение — `grad_diehl(x)[0]`.

**Задание 3 (1 балл): `fd_forward(f, x, i, h)` и `fd_central(f, x, i, h)`** — прямая и центральная разности по $i$-й координате:

$$
\frac{f(x+he_i)-f(x)}{h},\qquad\qquad \frac{f(x+he_i)-f(x-he_i)}{2h},
$$

где $e_i$ — $i$-й единичный вектор.

In [ ]:
def fd_forward(f, x, i, h):
    """Прямая разность по i-й координате."""
    # TODO: ваш код
    raise NotImplementedError


def fd_central(f, x, i, h):
    """Центральная разность по i-й координате."""
    # TODO: ваш код
    raise NotImplementedError

In [ ]:
# проверка задания 3: у q(a, b) = a^3 + b в точке (1, 0) ошибка усечения известна точно: 3h + h^2 у прямой, h^2 у центральной
q = lambda x: x[0] ** 3 + x[1]
xq, h = np.array([1.0, 0.0]), 1e-3
assert abs(fd_forward(q, xq, 0, h) - (3 + 3 * h + h * h)) < 1e-9, "прямая разность: (q(x + h e_i) - q(x)) / h"
assert abs(fd_central(q, xq, 0, h) - (3 + h * h)) < 1e-9, "центральная разность: (q(x + h e_i) - q(x - h e_i)) / (2h)"
assert abs(fd_central(q, xq, 1, h) - 1) < 1e-9, "по второй координате производная q равна 1: проверьте вектор e_i"
print("задание 3: ok")

**(а) Картинка 5 (1 балл): U-кривая.** Для `hs = np.logspace(-15, -1, 141)` посчитайте ошибки обеих формул относительно точного значения и нарисуйте `plot_ucurve(hs, [err_forward, err_central], labels)`; пунктиром на картинке отмечены $\sqrt\varepsilon\approx1.5\cdot10^{-8}$ и $\varepsilon^{1/3}\approx6\cdot10^{-6}$.

Ответьте: при каком $h$ ошибка каждой формулы наименьшая и какая это ошибка? Какая ветка буквы U — ошибка усечения, какая — округления, и какой наклон у каждой на log-log графике? Почему лучший $h$ у центральной разности больше?

*Полный балл:* картинка; лучшие $h$ и ошибки обеих формул по порядку величины; какая ветка — какая ошибка, наклоны веток; откуда берётся лучший $h$. *Половина:* картинка и числа без объяснения веток.

In [ ]:
# TODO: ваш код
# x_fd = np.array([1.0, 1.0]); exact = grad_diehl(x_fd)[0]; hs = np.logspace(-15, -1, 141)
# err_forward, err_central — массивы |разность - exact| по hs; лучший h — hs[np.argmin(...)]
# картинка: plot_ucurve(hs, [err_forward, err_central], ["прямая разность", "центральная разность"]); plt.show()


_Ваш ответ:_ TODO

**(б) (0.5 балла): CasADi.** Задайте ту же функцию символьно — через `z[0]`, `z[1]` и `ca.cos` — и получите градиент алгоритмическим дифференцированием, `ca.gradient`, как на семинаре 6. Посчитайте ошибку по первой координате в той же точке и добавьте её на картинку 5: `plot_ucurve(..., ad_error=err_ad)`.

Ответьте одной-двумя фразами: почему линия CasADi лежит ниже всей U-кривой и почему у неё нет $h$?

*Полный балл:* градиент CasADi, его ошибка, линия на картинке и объяснение. *Половина:* градиент без объяснения.

In [ ]:
z = ca.SX.sym("z", 2)                     # символьная точка: z[0] = u, z[1] = v
f_sym = ...                               # TODO: функция задачи 1 через z[0], z[1] и ca.cos
grad_ad = ca.Function("grad_ad", [z], [ca.gradient(f_sym, z)])
# TODO: ваш код — err_ad = |grad_ad(x_fd)[0] - exact| и картинка plot_ucurve(..., ad_error=err_ad)


_Ваш ответ:_ TODO

## Бонус (+1 балл). Dogleg своими руками

**Dogleg** (конспект §3.3) — приближённое решение подзадачи: ломаная из нуля в минимум модели вдоль $-g$, точку $p_U=-\dfrac{g^\top g}{g^\top Bg}\,g$, а оттуда в точку Ньютона $p_B=-B^{-1}g$. Шаг — точка, где ломаная выходит из круга $\lVert p\rVert\le\Delta$; если $p_B$ внутри круга — сама $p_B$; если круг меньше $\lVert p_U\rVert$ — точка $-\Delta\,g/\lVert g\rVert$. Ломаная имеет смысл только при $B\succ0$ (все собственные числа положительны), иначе верните `cauchy_point(g, B, Delta)`.

Напишите `dogleg(g, B, Delta)` в ячейке ниже и выполните следующую: она проверит функцию на подзадаче конспекта $g=(3,4)$, $B=\operatorname{diag}(1,5)$ (значения модели $-3.22$, $-5.21$, $-6.10$ при $\Delta=1,\ 2,\ 3.5$), нарисует подзадачу при $\Delta=2$ и запустит `trust_region` с `dogleg` на Розенброке — в конспекте 24 итерации. Ответьте одной фразой: чем dogleg хуже точного шага и чем лучше?

*Балл:* проверки проходят, картинка, число итераций на Розенброке и фраза о цене и точности dogleg. Бонус необязателен: если пропускаете, ячейки ниже оставьте как есть.

In [ ]:
# бонус (необязательно): напишите здесь dogleg(g, B, Delta)


In [ ]:
if "dogleg" in globals():         # бонус: выполняется, только если вы написали dogleg
    g_m, B_m = np.array([3.0, 4.0]), np.diag([1.0, 5.0])
    m_val = lambda p: g_m @ p + 0.5 * p @ B_m @ p
    for D, expected in ((1.0, -3.22), (2.0, -5.21), (3.5, -6.10)):
        p = dogleg(g_m, B_m, D)
        assert np.linalg.norm(p) <= D + 1e-9 and abs(m_val(p) - expected) < 1e-2, f"при Delta = {D} ожидалось m = {expected}, получено {m_val(p):.3f}"
    x_dl, k_dl, _, hist_dl = trust_region(rosen, rosen_grad, rosen_hess, X0_ROSEN, dogleg)
    print(f"бонус: проверки ok;  Розенброк с dogleg: итераций {k_dl}, отвергнуто {sum(not h[4] for h in hist_dl)}")
    plot_subproblem(g_m, B_m, 2.0, {"точка Коши": cauchy_point(g_m, B_m, 2.0), "dogleg": dogleg(g_m, B_m, 2.0),
                                    "точный шаг": tr_exact(g_m, B_m, 2.0)}); plt.show()
else:
    print("бонус пропущен")

_Бонус (необязательно):_ ваш ответ